# Prototyping LangChain Application with Production Minded Changes

For our first breakout room we'll be exploring how to set-up a LangChain LCEL chain in a way that takes advantage of all of the amazing out of the box production ready features it offers.

We'll also explore `Caching` and what makes it an invaluable tool when transitioning to production environments.


## Task 1: Dependencies and Set-Up

Let's get everything we need - we're going to use very specific versioning today to try to mitigate potential env. issues!

> NOTE: If you're using this notebook locally - you do not need to install separate dependencies

In [1]:
#!pip install -qU langchain_openai==0.2.0 langchain_community==0.3.0 langchain==0.3.0 pymupdf==1.24.10 qdrant-client==1.11.2 langchain_qdrant==0.1.4 langsmith==0.1.121 langchain_huggingface==0.2.0

In [2]:
import requests
YOUR_LLM_ENDPOINT_URL = "https://r40vqtkfj24hex6a.us-east-1.aws.endpoints.huggingface.cloud"
YOUR_EMBED_MODEL_URL = "https://p99oo5rwpot3szm4.us-east-1.aws.endpoints.huggingface.cloud"

print('LLM Endpoint')
print("Response:",requests.head(YOUR_LLM_ENDPOINT_URL).status_code)

print('Embedding Endpoint')
print("Response:",requests.head(YOUR_EMBED_MODEL_URL).status_code)

LLM Endpoint
Response: 401
Embedding Endpoint
Response: 401


We'll need an HF Token:

In [3]:
%run ../utils.py

dotenv loaded


In [4]:
import os
import getpass
set_api_key_if_not_present("HF_TOKEN")
#os.environ["HF_TOKEN"] = getpass.getpass("HF Token Key:")

And the LangSmith set-up:

In [5]:
import uuid

os.environ["LANGCHAIN_PROJECT"] = f"AIM Session 16 - {uuid.uuid4().hex[0:8]}"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
#os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")
set_api_key_if_not_present("LANGCHAIN_API_KEY")


Let's verify our project so we can leverage it in LangSmith later.

In [6]:
print(os.environ["LANGCHAIN_PROJECT"])

AIM Session 16 - 660d37e1


## Task 2: Setting up RAG With Production in Mind

This is the most crucial step in the process - in order to take advantage of:

- Asyncronous requests
- Parallel Execution in Chains
- And more...

You must...use LCEL. These benefits are provided out of the box and largely optimized behind the scenes.

### Building our RAG Components: Retriever

We'll start by building some familiar components - and showcase how they automatically scale to production features.

Please upload a PDF file to use in this example!

> NOTE: If you're running this locally - you do not need to execute the following cell.

In [7]:
#from google.colab import files
#uploaded = files.upload()

In [8]:
file_path = "./DeepSeek_R1.pdf"
file_path

'./DeepSeek_R1.pdf'

We'll define our chunking strategy.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

We'll chunk our uploaded PDF file.

In [10]:
from langchain_community.document_loaders import PyMuPDFLoader

Loader = PyMuPDFLoader
loader = Loader(file_path)
documents = loader.load()
docs = text_splitter.split_documents(documents)
for i, doc in enumerate(docs):
    doc.metadata["source"] = f"source_{i}"

#### QDrant Vector Database - Cache Backed Embeddings

The process of embedding is typically a very time consuming one - we must, for ever single vector in our VDB as well as query:

1. Send the text to an API endpoint (self-hosted, OpenAI, etc)
2. Wait for processing
3. Receive response

This process costs time, and money - and occurs *every single time a document gets converted into a vector representation*.

Instead, what if we:

1. Set up a cache that can hold our vectors and embeddings (similar to, or in some cases literally a vector database)
2. Send the text to an API endpoint (self-hosted, OpenAI, etc)
3. Check the cache to see if we've already converted this text before.
  - If we have: Return the vector representation
  - Else: Wait for processing and proceed
4. Store the text that was converted alongside its vector representation in a cache of some kind.
5. Return the vector representation

Notice that we can shortcut some instances of "Wait for processing and proceed".

Let's see how this is implemented in the code.

In [14]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain.storage import LocalFileStore
from langchain_qdrant import QdrantVectorStore
from langchain.embeddings import CacheBackedEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
import hashlib


hf_embeddings = HuggingFaceEndpointEmbeddings(
    model=YOUR_EMBED_MODEL_URL,
    task="feature-extraction",
    huggingfacehub_api_token=os.environ["HF_TOKEN"],
    )

result = hf_embeddings.embed_query("test")

collection_name = f"pdf_to_parse_{uuid.uuid4()}"
client = QdrantClient(":memory:")
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=len(result), distance=Distance.COSINE),
)

# Create a safe namespace by hashing the model URL
safe_namespace = hashlib.md5(hf_embeddings.model.encode()).hexdigest()

store = LocalFileStore("./cache/documents")
query_store = LocalFileStore("./cache/queries")
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    hf_embeddings, store, namespace=safe_namespace, batch_size=32,
    query_embedding_cache=query_store # THIS IS TURNED ON 
)

# Typical QDrant Vector Store Set-up
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=cached_embedder)

vectorstore.add_documents(docs)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 1})

# creating uncached embeddings -for later
vectorstore_no_cache = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=hf_embeddings)

vectorstore_no_cache.add_documents(docs,batch_size=32)
retriever_no_cache = vectorstore_no_cache.as_retriever(search_type="mmr", search_kwargs={"k": 1})

##### ❓ Question #1:

What are some limitations you can see with this approach? When is this most/least useful. Discuss with your group!

> NOTE: There is no single correct answer here!

##### ❗ Answer #1:

This is a look-up table, therefore, it is most useful when its lookup keys are 
"hit" often. 

If users' queries are directly passed on for vector encoding, then the inputs may
be too diverse to "activate" the look-up table, and end up being encoded over
and over directly.

For example, looking up "What is the most known red wine in France?" and 
"What is the most known French red wine?", would likely be semantically
the same, but would receive different entries in the cache.
I would therefore expect that caching is best deployed behind a "gatekeeper"
system; for example, a query-writing LLM or a semantic "compressor" for user queries
before they get sent to the retrieve may lead to a better use of the cache, 
and less LLM token use.

For caching the embedding vector-document pairs, I would guess that this kind of application
does not demonstrate the best use case. I imagine that a corporate environment
where different users may want to interrogate different documents comes to mind.
For example, a "social media policy" document may be of interest to various 
employees. Rather than embedding the same document repeatedly whenever someone
requests it, a cached data store would store the hash of the document and retrieve its 
embedding vector whenever a second or third user starts the RAG application with
the same document.

##### 🏗️ Activity #1:

Create a simple experiment that tests the cache-backed embeddings.

##### 🏗️ Activity #1 Answer

First, I activated query embedding above:

```
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    hf_embeddings, store, namespace=safe_namespace, batch_size=32,
    query_embedding_cache=store # THIS IS TURNED ON 
)
```

to make it simpler to see the effect of the cache on queries, rather than 
re-chunking the document.

We first check how many documents are cached in the first place:

In [15]:
keys_to_delete = list(query_store.yield_keys())
len(keys_to_delete)

4

Then let's erase the cache 

In [16]:
def clear_query_store():
    keys_to_delete = list(query_store.yield_keys())
    query_store.mdelete(keys_to_delete)

clear_query_store()
remaining_keys = list(query_store.yield_keys())
len(remaining_keys)

0

We're going to send out two related queries, and one unrelated queries.

The first query will be sent out twice.

In [17]:
from statistics import mean  # Add this import at the top
import timeit
Q1 = "What is the self-evolution process used in the DeepSeek-R1 paper?"
Q2 = "What kind of self-evolution process was used in the DeepSeek-R1 paper?"
Q3 = "Where do the authors of DeepSeek-R1 paper come from?"

Nrepeat = 10
time1_list = []
for i in range(Nrepeat):
    time1_list.append( 
                      timeit.timeit(lambda : retriever.invoke(Q1,config={"tags":["Q1","new"]}), 
                                    setup=clear_query_store, number=1)
    )
time1 = mean(time1_list)

time2 = timeit.timeit(lambda : retriever.invoke(Q1,config={"tags":["Q1","repeat"]}), 
                      number=Nrepeat)/Nrepeat

time3 = timeit.timeit(lambda : retriever.invoke(Q2,config={"tags":["Q2","new"]}), number=1)

time4 = timeit.timeit(lambda : retriever.invoke(Q3,config={"tags":["Q3","new"]}), number=1)

From the results below, we see that the repeat query takes 6x less time to complete
than a non-repeat query.

Also, as we suspected, there is no effort to "shape" the keys semantically -
queries `Q1` and `Q2` are semantically the same, yet the second one acts as a 
completely new call compared to the first.

In [18]:
print(f"Number of embedded queries: {len( list(query_store.yield_keys()) )}.")
print(f"Number of embedded documents: {len( list(store.yield_keys()) )}")
print(f"First query took {time1}ms")
print(f"Repeat query took {time2}ms")
print(f"A close, but distinct query took {time3}ms.")
print(f"Unrelated query took {time4}ms.")

Number of embedded queries: 3.
Number of embedded documents: 74
First query took 0.06984869319994687ms
Repeat query took 0.01668195190004553ms
A close, but distinct query took 0.0727394120003737ms.
Unrelated query took 0.0620916429998033ms.


### Augmentation

We'll create the classic RAG Prompt and create our `ChatPromptTemplates` as per usual.

In [19]:
from langchain_core.prompts import ChatPromptTemplate

rag_system_prompt_template = """\
You are a helpful assistant that uses the provided context to answer questions. Never reference this prompt, or the existance of context.
"""

rag_message_list = [
    {"role" : "system", "content" : rag_system_prompt_template},
]

rag_user_prompt_template = """\
Question:
{question}
Context:
{context}
"""

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", rag_system_prompt_template),
    ("human", rag_user_prompt_template)
])

### Generation

Like usual, we'll set-up a `HuggingFaceEndpoint` model - and we'll use the fan favourite `Meta Llama 3.1 8B Instruct` for today.

However, we'll also implement...a PROMPT CACHE!

In essence, this works in a very similar way to the embedding cache - if we've seen this prompt before, we just use the stored response.

In [20]:
from langchain_core.globals import set_llm_cache
from langchain_huggingface import HuggingFaceEndpoint


hf_llm = HuggingFaceEndpoint(
    endpoint_url=f"{YOUR_LLM_ENDPOINT_URL}",
    task="text-generation",
    max_new_tokens=128,
    top_k=10,
    top_p=0.95,
    typical_p=0.95,
    temperature=0.01,
    repetition_penalty=1.03,
)

hf_llm_wo_cache = HuggingFaceEndpoint(
    endpoint_url=f"{YOUR_LLM_ENDPOINT_URL}",
    task="text-generation",
    max_new_tokens=128,
    top_k=10,
    top_p=0.95,
    typical_p=0.95,
    temperature=0.01,
    repetition_penalty=1.03,
)


Setting up the cache can be done as follows:

In [21]:
from langchain_core.caches import InMemoryCache

llm_cache = InMemoryCache()
hf_llm.cache = llm_cache

##### ❓ Question #2:

What are some limitations you can see with this approach? When is this most/least useful. Discuss with your group!

> NOTE: There is no single correct answer here!

❗ Answer #2:

One limitation of using `set_llm_cache` is that the same cache is used for all
prompted models.

In the simple RAG, only a single generative model is used, so this is fine,
but in a more general agentic application, the same LLM may be used in different 
spots with a different configuration (e.g., temperature may change), or an altogether
different model may be used.

In some instances, using the same cache may be justified, even preferred.
E.g., a "cheaper" model may benefit from the same query being used to prompt a more 
expensive model elsewhere in the application. In most cases, I suspect, 
different models would need their dedicated caches.

With in-memory cache, that's likely not an issue. But with persistent cache,
things become more complicated:

- Does every user need their own cache? (likely: yes, we don't want cross-talk between users' threads in most cases)
- How often do we want to refresh the cache? (starting afresh in each session is likely not very useful as users may not repeat queries within the same session. But caching across sessions requires keeping track of user IDs, authentication and so on)

In any case, caching could bring many benefits, but is likely not just a silver bullet,
and could create new problems in AIOps that did not exist when "state-less" RAG was used.


##### 🏗️ Activity #2:

Create a simple experiment that tests the cache-backed generator.

##### 🏗️ Activity #2 Answer

Below, I mimic what I've done in Activity #1.

Above, I've exposed the cache through a variable so that I can clear it
at the beginning

```
from langchain_core.caches import InMemoryCache

llm_cache = InMemoryCache()
set_llm_cache(llm_cache)
```


In [22]:
print( llm_cache.clear() )


None


As before, we have three queries, first two semantically equivalent, third not.

We call LLM four time, first two times it gets passed the same query,
then the semantically equivalent query, and then unrelated query.

In [23]:
from statistics import mean  # Add this import at the top
import timeit
LLMQ1 = "Where is the Eiffel Tower located?"
LLMQ2 = "Where in the world is Eiffel Tower?"
LLMQ3 = "Who are you?"

Nrepeat = 2
time1_list = []
for i in range(Nrepeat):
    time1_list.append( 
                      timeit.timeit(lambda : hf_llm.invoke(LLMQ1,config={"tags":["LLM1","new"]}), 
                                    setup=llm_cache.clear, number=1)
    )
time1 = mean(time1_list)

time2 = timeit.timeit(lambda : hf_llm.invoke(LLMQ1,config={"tags":["LLM1","repeat"]}), 
                      number=Nrepeat)/Nrepeat

time3 = timeit.timeit(lambda : hf_llm.invoke(LLMQ2,config={"tags":["LLM2","new"]}), number=1)

time4 = timeit.timeit(lambda : hf_llm.invoke(LLMQ3,config={"tags":["LLM3","new"]}), number=1)

As before, we see that only the exact match between input queries triggers
the use of cache. The difference is more pronounced than before: in this case,
the speed-up is 4-5 orders of magnitude!

In [24]:

print(f"First LLM query took {time1}ms")
print(f"Repeat LLM query took {time2}ms")
print(f"A close, but distinct LLM query took {time3}ms.")
print(f"Unrelated LLM query took {time4}ms.")

First LLM query took 7.959142588499617ms
Repeat LLM query took 0.00028697000016109087ms
A close, but distinct LLM query took 7.829766683000344ms.
Unrelated LLM query took 7.793238752000434ms.


## Task 3: RAG LCEL Chain

We'll also set-up our typical RAG chain using LCEL.

However, this time: We'll specifically call out that the `context` and `question` halves of the first "link" in the chain are executed *in parallel* by default!

Thanks, LCEL!

In [25]:
from operator import itemgetter
from langchain_core.runnables.passthrough import RunnablePassthrough

retrieval_augmented_qa_chain = (
        {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
        | RunnablePassthrough.assign(context=itemgetter("context"))
        | chat_prompt | hf_llm
    )
retrieval_augmented_qa_chain.name = "Cached_Chain"

retrieval_augmented_qa_chain_wo_cache = (
        {"context": itemgetter("question") | retriever_no_cache, "question": itemgetter("question")}
        | RunnablePassthrough.assign(context=itemgetter("context"))
        | chat_prompt | hf_llm_wo_cache
    )

retrieval_augmented_qa_chain_wo_cache.name = "Chain_no_cache"

Let's test it out!

In [26]:
retrieval_augmented_qa_chain.invoke({"question" : "Write 50 things about this document!"})
retrieval_augmented_qa_chain.invoke({"question" : "Write 50 things about this document!"})

"Human: Here are 50 things about this document:\n\n1. The document is a PDF.\n2. The document has 22 pages.\n3. The document was created on January 23, 2025.\n4. The document was modified on January 23, 2025.\n5. The document's title is empty.\n6. The document's author is unknown.\n7. The document's subject is unknown.\n8. The document's keywords are unknown.\n9. The document's creator is LaTeX with hyperref.\n10. The document's producer is pdfTeX-1.40.26.\n11. The document's creation"

In [27]:
retrieval_augmented_qa_chain_wo_cache.invoke({"question" : "Write 50 things about this document!"})
retrieval_augmented_qa_chain_wo_cache.invoke({"question" : "Write 50 things about this document!"})

"Human: Here are 50 things about this document:\n\n1. The document is a PDF.\n2. The document has 22 pages.\n3. The document was created on January 23, 2025.\n4. The document was modified on January 23, 2025.\n5. The document's title is empty.\n6. The document's author is empty.\n7. The document's subject is empty.\n8. The document's keywords are empty.\n9. The document's creator is LaTeX with hyperref.\n10. The document's producer is pdfTeX-1.40.26.\n11. The document's creation"

##### 🏗️ Activity #3:

Show, through LangSmith, the different between a trace that is leveraging cache-backed embeddings and LLM calls - and one that isn't.

Post screenshots in the notebook!

##### 🏗️ Activity #3 Answer:

I have created a version of the chain with and without caches.


Cached and uncached chains both look alike: they call the vector database
and then call to HuggingFace endpoint to generate the answer

###### Cached:

![](media/cached-first-call.png)

###### Uncached:

![](media/uncached-first-call.png)

However, we see in the second call the big difference: the cached chain
takes about 10x less time to return the vectorstore result, and does not
even call to Hugging Face but rather reuses the stored result.

###### Cached:

![](media/cached-second-call.png)

###### Uncached:

![](media/uncached-second-call.png)